# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR\^2 mass spectrometry dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata and inspect metadata object
dataset = mlc.Dataset(croissant_url)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print("\nDataset citation:")
print(dataset.metadata.citeAs)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Let's enumerate all top-level record sets, then for one, print its fields and columns by their `@id`.

In [ ]:
# List all available record sets and their @id and .name
print("Available record sets (by @id):\n")
record_sets = list(dataset.metadata.recordSet)
for record_set in record_sets:
    print(f"  {record_set['@id']}  |  {record_set.get('name', '<no name>')}")

# Choose the main record set (usually protein table); pick the first
main_record_set_id = record_sets[0]['@id'] if record_sets else None

print(f"\nInspecting structure for main record set: {main_record_set_id}")
fields = record_sets[0]['field'] if record_sets and 'field' in record_sets[0] else []

if fields:
    print("\nFields in this record set (by @id):")
    for f in fields:
        print(f"  {f['@id']}  |  {f.get('name', '<no name>')}")
else:
    print('No fields found in chosen record set.')

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. We use record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into pandas DataFrames by their @id
# (You can add or remove record set IDs as appropriate for your exploration)
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
dataframes = {}

for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded {len(df)} records for record set {rset_id}")
    print(df.columns.tolist(), '\n')

# Preview main DataFrame
main_df = dataframes[main_record_set_id]
print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
We'll:

- Select a numeric field by its `@id` (e.g. peptide counts, molecular weight, abundance)
- Filter and normalize it
- Optionally group by a categorical field (e.g. `description` or sample condition)

Replace the placeholder field IDs with those relevant for this dataset.

In [ ]:
# --- Set the field IDs for numeric and grouping analysis ---
# Example field @ids: use actual ones from the overview above

# Suppose there is a field '@id': 'field:peptide_count' and 'field:molecular_weight'
numeric_field_id = None
group_field_id = None
for f in dataset.metadata.recordSet[0]['field']:
    id_candidate = f['@id']
    name = f.get('name', '')
    # Try to pick a numeric field (e.g. peptide count, molecular weight, etc.)
    if not numeric_field_id and ('peptide' in name.lower() or 'count' in id_candidate.lower()):
        numeric_field_id = id_candidate
    if not group_field_id and (('description' in name.lower()) or ('sample' in name.lower())):
        group_field_id = id_candidate

if not numeric_field_id:
    # Fallback: pick first non-text field
    for f in dataset.metadata.recordSet[0]['field']:
        if f.get('dataType', '').lower() in ['integer', 'float', 'number']:
            numeric_field_id = f['@id']
            break

print(f"Using numeric field @id for EDA: {numeric_field_id}")
print(f"Using group field @id (if available): {group_field_id}")

# Check that our main DataFrame has those fields
df = main_df.copy()

if numeric_field_id and numeric_field_id in df.columns:
    # Try to coerce to float
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in main dataframe columns:", df.columns.tolist())

## 5. Visualization
Visualize the distribution of the numeric field and optionally relationships between grouped categories.

You might need to adjust field IDs used in the plot code based on fields available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped data is available, plot group means
    if ('grouped_df' in locals()) and (grouped_df.shape[0] > 1):
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field selected or not present for plotting.")

## 6. Conclusion
In this notebook, we demonstrated how to explore a multi-record-set FAIR<sup>2</sup> mass spectrometry dataset with `mlcroissant`:

- We loaded metadata and data using Croissant schema URLs and referenced all fields by their `@id`.
- We listed available record sets and fields, and loaded records into pandas DataFrames for further analysis.
- We performed basic filtering, normalization, grouping, and visualized key distributions.

This process can be adapted for any Croissant-compatible dataset by adjusting the `@id` variables for field and record set references. For more advanced analyses, experiment with the data structure using the field and record set metadata.